## Install & Import Libraries

In [1]:
!pip install -q accelerate transformers

In [2]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.4 MB/s eta 0:00:00


In [3]:
from huggingface_hub import login
login("Put your key here")

##  Step 3: Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [4]:
import pandas as pd
MentalQA = pd.read_excel('Entities&Relations_MentalQA_500_data.xlsx')

In [5]:
MentalQA.head()

,question,answer,final_QT,final_AS
0,هل يعتبر الخوف من عدم الإنجاب مستقبلاً حالة عا...,سد تى نعم ان هناك العديد من القلق النفسى المرت...,"['A', 'D']","['1', '2']"
1,من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و...,يجب العرض علي طبيب اختصاصي امراض نفسية سوف يقو...,"['B', 'E']",['2']
2,السلام عليكم مشكلتي تقتصر على تكرار كلمة معينة...,الوسواس القهري هو من الامراض الشاءعة ويكون في ...,"['A', 'E']","['1', '2']"
3,اكتئاب وفوبيا من المجتمع وانعزال وانطوائية وتع...,العلاج النفسي المعرفي السلوكي يعطي نتائج جيدة ...,"['B', 'D']",['1']
4,عمري 37 سنة وحامل في الشهر الثامن . منذ الشهر ...,ثابري مع طبيبك واخبريه بمخاوفك والتزمي بالجوان...,"['A', 'B', 'D']","['2', '3']"


In [6]:
MentalQA.columns

Index(['question', 'answer', 'final_QT', 'final_AS'], dtype='object')

In [7]:
df = MentalQA.copy()

In [8]:
df_q = df[["question"]]

## Prompt

In [9]:
def build_prompt(question):
    return f"""

أنت خبير في المعالجة اللغوية للنصوص الطبية باللغة العربية، ومهمتك تنفيذ عملية الوسم الدلالي
(Named Entity Recognition - NER)
.لاستخراج كيانات طبية محددة من استفسارات يطرحها المرضى في سياق الصحة النفسية

تتمثل مهمتك في تحديد واستخراج الكيانات التالية فقط، كما وردت نصًا في الاستفسار:

- Symptom: الأعراض النفسية أو السلوكية الظاهرة
- Diagnosis: الحالات أو الاضطرابات النفسية المشخّصة أو المشار إليها
- Treatment: طرق أو أنواع العلاج غير الدوائية (مثل: العلاج السلوكي المعرفي، الاستشارة النفسية)
- Drug: أسماء الأدوية أو العقاقير
- Dosage: معلومات عن الجرعة (مثلًا: مرتين في اليوم، 50 ملغ)
- Duration: مدة الاستخدام أو استمرار الحالة (مثلًا: منذ أسبوع، لمدة 3 أشهر)
- Strength: قوة أو تركيز الدواء (مثلًا: 500، 1.0)
- Unit: وحدة قياس التركيز (مثلًا: ملغ، g، ml)
- Risk: مؤشرات على وجود خطر على السلامة النفسية أو الجسدية (مثل الانتحار أو إيذاء النفس)

تعليمات مهمة:
- استخرج فقط الكيانات المذكورة بشكل صريح في نص الاستفسار.
- استخدم الصياغة الحرفية كما وردت، سواء كانت بالعربية أو الإنجليزية.
- اكتب النتيجة بصيغة قائمة واحدة تحتوي على جميع الكيانات، بهذا التنسيق:
[{{"Symptom": "...", "Diagnosis": "...", "Treatment": "...", "Drug": "...", "Dosage": "...", "Duration": "...", "Strength": "...", "Unit": "...", "Risk": "..."}}]
- إذا لم يُذكر كيان معين، اجعل قيمته: ""
- لا تضف أي شرح أو تعليق إضافي — فقط أعد الكيانات بالتنسيق المطلوب.

أمثلة:

مثال ١:
السؤال: كيف انتقل من دواء السيروكسات الذي اتناول منه جرعة70ملغم يوميا منذ عام الى السيبرالكس لاتفادى الاعراض الانحسابية علما ان الطبيب المعالج طلب مني ان استبدل السيروكسات بالسيبرالكس
الإخراج:
[{{"Symptom": "", "Diagnosis": "", "Treatment": "استبدل", "Drug": "السيروكسات، السيبرالكس", "Dosage": "70ملغم يوميا", "Duration": "منذ عام", "Strength": "70", "Unit": "ملغم", "Risk": ""}}]

مثال ٢:
السؤال: بدأت حديثا بتناول دواء سيروكسات 20 لمعالجة الرهاب الاجتماعي بمقدار نصف حبة يوميا لكن ظهرت لدي مشكلة كثرة التثاؤب ,هل ستبقى هذه المشكلة طيلة فترة العلاج او فقط في البداية ؟ كيف اتخلص منها ؟
الإخراج:
[{{"Symptom": "كثرة التثاؤب", "Diagnosis": "الرهاب الاجتماعي", "Treatment": "معالجة", "Drug": "سيروكسات", "Dosage": "نصف حبة يوميا", "Duration": "", "Strength": "20", "Unit": "", "Risk": ""}}]

مثال ٣:
السؤال: من سنه تقريبا و انا أذي نفسي ب اكثر من طريقة و ما اعرف كيف اتخلص من ذي العادة, و بدت تجيني افكار بإنهاء حياتي و حاولت انتحر باكثر من مرة و أكثر من طريقة
الإخراج:
[{{"Symptom": "أذي نفسي", "Diagnosis": "", "Treatment": "", "Drug": "", "Dosage": "", "Duration": "من سنه", "Strength": "", "Unit": "", "Risk": "أذي نفسي، حاولت انتحر، افكار بإنهاء حياتي"}}]

مثال ٤:
السؤال: هل الانتحار عن طريق حبوب المنوم يجعلني اشعر بالراحة ولا يكون مؤلم؟
الإخراج:
[{{"Symptom": "", "Diagnosis": "", "Treatment": "", "Drug": "حبوب المنوم", "Dosage": "", "Duration": "", "Strength": "", "Unit": "", "Risk": "الانتحار، الانتحار عن طريق حبوب المنوم"}}]

الآن استخرج الكيانات من المثال التالي:

السؤال: {question}
"""


### Zero-shot

In [9]:
def build_prompt(question):
    return f"""

أنت خبير في المعالجة اللغوية للنصوص الطبية باللغة العربية، ومهمتك تنفيذ عملية الوسم الدلالي
(Named Entity Recognition - NER)
.لاستخراج كيانات طبية محددة من استفسارات يطرحها المرضى في سياق الصحة النفسية

تتمثل مهمتك في تحديد واستخراج الكيانات التالية فقط، كما وردت نصًا في الاستفسار:

- Symptom: الأعراض النفسية أو السلوكية الظاهرة
- Diagnosis: الحالات أو الاضطرابات النفسية المشخّصة أو المشار إليها
- Treatment: طرق أو أنواع العلاج غير الدوائية (مثل: العلاج السلوكي المعرفي، الاستشارة النفسية)
- Drug: أسماء الأدوية أو العقاقير
- Dosage: معلومات عن الجرعة (مثلًا: مرتين في اليوم، 50 ملغ)
- Duration: مدة الاستخدام أو استمرار الحالة (مثلًا: منذ أسبوع، لمدة 3 أشهر)
- Strength: قوة أو تركيز الدواء (مثلًا: 500، 1.0)
- Unit: وحدة قياس التركيز (مثلًا: ملغ، g، ml)
- Risk: مؤشرات على وجود خطر على السلامة النفسية أو الجسدية (مثل الانتحار أو إيذاء النفس)

تعليمات مهمة:
- استخرج فقط الكيانات المذكورة بشكل صريح في نص الاستفسار.
- استخدم الصياغة الحرفية كما وردت، سواء كانت بالعربية أو الإنجليزية.
- اكتب النتيجة بصيغة قائمة واحدة تحتوي على جميع الكيانات، بهذا التنسيق:
[{{"Symptom": "...", "Diagnosis": "...", "Treatment": "...", "Drug": "...", "Dosage": "...", "Duration": "...", "Strength": "...", "Unit": "...", "Risk": "..."}}]
- إذا لم يُذكر كيان معين، اجعل قيمته: ""
- لا تضف أي شرح أو تعليق إضافي فقط أعد الكيانات بالتنسيق المطلوب.



**السؤال**:

{question}


 اكتب فقط قائمة الكيانات، بدون أي شرح أو تعليق إضافي.

"""

## Model

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "ALLaM-AI/ALLaM-7B-Instruct-preview"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name) # use_auth_token=True
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
   # use_auth_token=True
)

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.23MB            

tokenizer.model: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [11]:
def call_allam(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output.replace(prompt, "").strip()

In [12]:
results_dict = {}

for i, row in df_q.iterrows():
    key = row['question']
    results_dict[key] = {
        "question": row["question"]
    }

In [13]:
import json
import time

batch_size = 100
save_every = 10
sleep_time = 0.2
results_path = "allam-results.json"
total_rows = len(df_q)

In [14]:
from tqdm.auto import tqdm
import sys

# Loop through dataset in batches
for batch_start in range(0, total_rows, batch_size):
    batch_end = min(batch_start + batch_size, total_rows)
    print(f"\nProcessing batch {batch_start} → {batch_end}")

    batch_df = df_q.iloc[batch_start:batch_end].copy()

    # Use tqdm as a variable to avoid re-instantiation
    progress_bar = tqdm(
        batch_df.iterrows(),
        total=len(batch_df),
        desc=f"Batch {batch_start // batch_size + 1}",
        leave=True,
        position=0,
        dynamic_ncols=True,
        file=sys.stdout,     # explicitly route output
        mininterval=0.5      # smooth updates
    )

    for i, row in progress_bar:
        key = row["question"]

        if "Allam Output" in results_dict.get(key, {}):
            continue

        try:
            prompt = build_prompt(row["question"])
            allam_output = call_allam(prompt)
            time.sleep(sleep_time)
        except Exception as e:
            allam_output = f"Allam Error: {e}"
            with open(results_path, "w", encoding="utf-8") as f:
                json.dump(results_dict, f, ensure_ascii=False, indent=2)
            print(f"\nError at row {i + batch_start}: {e}")
            break

        if key not in results_dict:
            results_dict[key] = {"question": row["question"]}

        results_dict[key]["Allam Output"] = allam_output

        # Save after each successful row
        with open(results_path, "w", encoding="utf-8") as f:
            json.dump(results_dict, f, ensure_ascii=False, indent=2)


Processing batch 0 → 100


Batch 1:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 100 → 200


Batch 2:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 200 → 300


Batch 3:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 300 → 400


Batch 4:   0%|          | 0/100 [00:00<?, ?it/s]


Processing batch 400 → 499


Batch 5:   0%|          | 0/99 [00:00<?, ?it/s]

## Results

In [15]:
allam_df = pd.DataFrame(results_dict.values())

In [16]:
allam_df.tail(20)

,question,Allam Output
479,اشعر بالحزن الداءم و انعدام التفاؤل و كثرة الر...,"**الإجابة**: \n\n[{""Symptom"": ""الحزن الداءم و..."
480,"يا دكتور أنا عندي 17 سنة, فيه مشكله غيرت طريقة...","**الإجابة**:\n[{""Symptom"": ""مش أنا, وحاسه اني ..."
481,السلام عليكم. ارجو ان تساعدني في حل هذه المشكل...,"الإجابة:\n\n[{""Drug"": ""اكل الاظافر"", ""Dosage"":..."
482,اعاتي من افكار تسلطيه وقهريه دائمه بعض الاحيان...,"```json\n[{""Symptom"": ""افات تسلسيه وقهريه دائم..."
483,"السلام عليكم دكتور, انا اشعر بالتوتر كثيرا فير...","النتيجة:\n\n```json\n[{""Symptom"": ""التوتر"", ""D..."
484,أعاني من رهاب إجتماعي حيت أجد صعوبة مثلا في ال...,"**الإجابة**:\n[{""Symptom"": ""رهاب إجتماعي"", ""Di..."
485,اعاني من الخوف والقلق و التفكير المسيطر وارتفا...,"**الإجابة**:\n\n[{""Symptom"": ""الخوف و القلق، ا..."
486,أعاني من فقدان للتركيز لا استطيع أن احدد مداه ...,"**الإجابة**:\n[{""Symptom"": ""فقدان للتركيز"", ""D..."
487,السلام عليكم انا شاب ابلغ من العمر 25 سنة اعان...,"**الإجابة**:\n[{""Symptom"": ""توتور بدون اي سبب ..."
488,احس بانفصال عن الواقع ودائما ابلع ريقي باستمرا...,"**الإجابة**:\n\n[{""Symptom"": ""احس بانفصال عن ا..."


In [17]:
from google.colab import files

allam_df.to_excel("allam_df_zeroshot.xlsx", index=False)
files.download("allam_df_zeroshot.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>